# MiMo-V2.6-Flash-RL on 4x CMP 170HX (SM80) — 4card-pp4, vLLM

| Metric | Value |
|---|---|
| Decode, c=1 | *run in progress — pending* |
| Prefill | *pending* |
| TTFT | *pending* |

Status: the quad was reconsolidated for this run; bring-up and benchmark are in flight.
This skeleton is committed before the first expensive load so the run is auditable from the start.

```bash
docker pull vllm/vllm-openai:mimov25-cu129
```

In [ ]:
# --- Status cell ---
EXPERIMENT = "%s"
RESULTS_DIR = "../results/" + EXPERIMENT
RECEIPTS = RESULTS_DIR + "/receipts"
LIVE = False

print(f"experiment : {EXPERIMENT}")
print(f"LIVE       : {LIVE}")
print("status     : skeleton committed before load; receipts land as gates pass.")

In [ ]:
# --- Helpers: receipt loader and table renderer ---
import json, os
from IPython.display import display, Markdown


def receipt(*parts):
    path = os.path.join(RECEIPTS, *parts)
    with open(path) as fh:
        return json.load(fh)


def render_table(headers, rows):
    lines = ["| " + " | ".join(str(h) for h in headers) + " |",
             "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        lines.append("| " + " | ".join(str(c) for c in row) + " |")
    display(Markdown(chr(10).join(lines)))


def fmt(v, nd=2):
    return "untested (pending)" if v is None else f"{v:,.{nd}f}"


assert not LIVE, "commit this notebook with LIVE = False"
print("receipts:", len(os.listdir(RECEIPTS)) if os.path.isdir(RECEIPTS) else 0, "entries")

## 1. TL;DR

*(to fill after the benchmark: verdict, headline tok/s, aggregate, prefill, TTFT, chart)*

In [ ]:
pins = {
    "model": "MiMo-V2.6-Flash-RL",
    "checkpoint": "XiaomiMiMo/MiMo-V2.6-Flash-RL",
    "checkpoint_revision": "5711b268169967567844e1e560e8a3966da959b1",
    "checkpoint_bytes": None,  # filled at the verify gate from model.safetensors.index.json
    "quantization": "FP8 e4m3, weight_block_size 128x128, stored as MXFP4 blocks",
    "runtime_source": None,    # pinned after the import probe decides the image
    "topology": "PP4",
    "max_model_len": None,
    "gpu_memory_utilization": 0.85,
    "cards": "4x CMP 170HX (SM80, 64 GiB each), no NVLink, no P2P",
    "power_cap": "180 W bench cap (matches the 2026-09-05 GLM protocol)",
    "measured_utc": None,
}
for k, v in pins.items():
    print(f"{k:24s} {v if v is not None else chr(39)+chr(39)}")

### Protocol

Mirrors the 2026-09-05 GLM-5.3-Flash protocol so lanes stay comparable:

| | P1 | P2 |
|---|---|---|
| Sampling | temperature 0 (greedy) | temperature 1.0, top_p 0.95 (vendor-recommended for MiMo-V2.6), `ignore_eos` |
| Output tokens | 512 | 512 |
| Repetitions | 3, median reported | 5, median reported (first rep cold) |
| Workloads | code, json, counting, math, prose | one fixed long-form prompt |
| Degeneracy guard | repeat-guard flags collapsed completions; flagged cells not read as throughput | none — read beside P1 |

Token counts come from the final `usage` object of each response, never from stream events.

## 2. Visible results

*(to fill as gates pass: boot/load gate, functional gate, decode c=1, concurrency, prefill/TTFT, stability/power/thermals)*

In [ ]:
render_table(
    ["Measurement", "Status"],
    [
        ["import probe (mimo_v2 arch on SM80)", "pending"],
        ["boot gate (PP4 load, KV pool, deterministic greedy)", "pending"],
        ["P1 c=1 decode per workload", "pending"],
        ["P2 c=1 decode", "pending"],
        ["concurrency sweep", "pending"],
        ["prefill / TTFT", "pending"],
        ["stability, power, thermals", "pending"],
    ],
)

## 3. Reproduce

### Download and verify the weights

```bash
pip install -U huggingface_hub
hf download XiaomiMiMo/MiMo-V2.6-Flash-RL --local-dir /library/models/mimo-v2.6-flash-rl
```

```python
import json, pathlib
root = pathlib.Path("/library/models/mimo-v2.6-flash-rl")
idx = json.load(open(root / "model.safetensors.index.json"))
files = sorted(set(idx["weight_map"].values()))
total = sum((root / f).stat().st_size for f in files)
print(f"shards: {len(files)}  total_bytes: {total:,}")
```

### Import probe

Before any GPU time, confirm the runtime registers `MiMoV2ForCausalLM` and that the
quantization path (fp8 e4m3 / MXFP4 store) has an SM80-compatible kernel or a
weight-only dequant fallback. 15 minutes here kills bad candidates for free.

### Launch

```bash
docker run --gpus all --shm-size 32g -v /library/models:/models -p 8000:8000 \
  vllm/vllm-openai:mimov25-cu129 \
  --model /models/mimo-v2.6-flash-rl \
  --pipeline-parallel-size 4 \
  --trust-remote-code \
  --gpu-memory-utilization 0.85 \
  --max-model-len 32768 \
  --reasoning-parser mimo
```

PP, never TP: no NVLink and no P2P on these cards; TP4 measured 6.6x worse on
the same fabric during the 2026-09 experiments. `--gpu-memory-utilization 0.85`
reserves headroom for a drafter if speculative decode is added later; cudagraphs
stay on (`--enforce-eager` measured 8-10 tok/s on this stack — never).

### First request

```bash
curl -s http://localhost:8000/v1/chat/completions -H 'Content-Type: application/json' -d '{
  "model": "mimo-v2.6-flash",
  "messages": [{"role": "user", "content": "Edit me before sending."}],
  "max_tokens": 256,
  "temperature": 0
}' | jq '{content: .choices[0].message.content, usage: .usage}'
```

### Bench

P1/P2 scripts and raw receipts land under `results/2026-09-22-mimo-v2.6-flash-4card-pp4-vllm/receipts/`.

### Attribution

- Model: Xiaomi MiMo team, MiMo-V2.6 release (MIT), 2026-09-22.
- PP-on-170HX serving shape follows the club's 2026-08/09 DeepSeek-V4-Flash and
  GLM-5.3-Flash notebooks and the community 4x 170HX DeepSeek recipe that
  established PP-not-TP on this fabric.
- Power cap, QC gates, and the speculative-decode posture come from the club's
  CMP 170HX serving strategy notes.

## 4. Appendix

*(superseded attempts, negative results, and raw boot logs land here)*